# Automated Acceptance Gate

Notebook 2 audited the model by hand. This notebook turns those judgment calls into an automated gate (`src/acceptance_gate.py`) that grades any fitted MMM without an analyst in the loop.

| Check | Fails or warns when | Why |
|---|---|---|
| Divergences | any divergent transition | the sampler missed part of the posterior |
| Max R-hat | > 1.01 | chains disagree |
| Min bulk ESS | < 400 | too few effective draws |
| Test MAPE | warn > 25%, fail > 40% | poor out-of-sample fit |
| Degenerate baseline | P(baseline < 0) > 5% | media is absorbing organic demand |
| Baseline share | warn if mean < 10% | baseline is set by the prior, not the data |
| Media share | P(media > 100% of sales) > 5% | impossible decomposition |
| ROAS interval width | warn if median (97% − 3%) / mean > 3 | show ranges, not rankings |
| Identifiability | warn if > 50% of channels are prior-driven | most effects are unmeasured |

**Tiers:** A = all pass (publish) · B = warnings only (publish with caveats) · C = any failure (do not publish; route to review).

Thresholds live in one `Thresholds` object so they can be tuned against analyst judgment. Every check has a unit test proving it fires (`tests/test_acceptance_gate.py`).

In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("src")

import pandas as pd
from acceptance_gate import Thresholds, evaluate_mmm
from data_prep import prepare
from mmm_specs import INTERCEPT_PRIORS, fit_spec

weekly, channels, controls = prepare("stunnerz_skateboards_simulated_data.csv")
split = int(len(weekly) * 0.8)
train, test = weekly.iloc[:split], weekly.iloc[split:]
X_train, y_train = train.drop(columns="sales"), train["sales"]
X_test, y_test = test.drop(columns="sales"), test["sales"]
SEED = sum(map(ord, "stunnerz_mmm"))

## Fit the three audit specifications and grade each one

In [2]:
reports = {}
for name in INTERCEPT_PRIORS:
    fit = fit_spec(name, X_train, y_train, X_test, channels, controls, seed=SEED)
    reports[name] = evaluate_mmm(fit["mmm"], fit["X_train"], y_train, y_test, fit["test_pred"], fit["channels"])

summary = pd.DataFrame({
    name: {**{c.name: f"{c.status} ({c.value})" for c in r.checks}, "TIER": r.tier}
    for name, r in reports.items()
})
summary

Sampling: [adstock_alpha, gamma_control, gamma_fourier, intercept_contribution, saturation_beta, saturation_lam, y, y_sigma]


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (4 chains in 1 job)


NUTS: [intercept_contribution, adstock_alpha, saturation_lam, saturation_beta, gamma_control, gamma_fourier, y_sigma]


Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 256 seconds.


...

Sequential sampling (4 chains in 1 job)


NUTS: [intercept_contribution, adstock_alpha, saturation_lam, saturation_beta, gamma_control, gamma_fourier, y_sigma]


Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 222 seconds.


Sampling: [y]


,A. Original priors,B. Positive baseline,C. Brand search as control
divergences,FAIL (1),PASS (0),PASS (0)
max R-hat,PASS (1.0),PASS (1.0),PASS (1.0)
min bulk ESS,PASS (2517),PASS (1786),PASS (2511)
test MAPE,PASS (0.217),WARN (0.26),WARN (0.369)
degenerate baseline,FAIL (0.951),PASS (0.0),PASS (0.0)
baseline share,WARN (-0.276),WARN (0.054),WARN (0.035)
media share <= 100%,FAIL (0.624),PASS (0.004),PASS (0.0)
ROAS interval width,WARN (3.07),WARN (3.07),WARN (3.24)
identifiability,WARN (0.92),WARN (0.83),WARN (0.9)
TIER,C,B,B


### Reading the result

- **Spec A is rejected (Tier C)** even though it has the best test MAPE. The gate catches what out-of-sample error alone cannot: a negative baseline and media explaining more than all of sales.
- **Specs B and C are not clean passes.** Their baselines sit near zero and most channels are prior-driven, so at best they publish with caveats. That matches the audit: the data cannot separate baseline from media without an experiment.

## Full report for Spec B

In [3]:
report_b = reports["B. Positive baseline"]
print("Tier:", report_b.tier)
report_b.to_frame()

Tier: B


,status,value,threshold,detail
name,,,,
divergences,PASS,0.000,<= 0,Divergent transitions mean the sampler could n...
max R-hat,PASS,1.000,<= 1.01,Chains disagree if R-hat is above threshold.
min bulk ESS,PASS,1786.000,>= 400,Too few effective draws to summarize the poste...
test MAPE,WARN,0.260,"warn > 0.25, fail > 0.4",Necessary but not sufficient: a model can pred...
degenerate baseline,PASS,0.000,P(baseline < 0) <= 0.05,A negative baseline means media is absorbing o...
baseline share,WARN,0.054,warn < 0.1,A baseline near zero is usually set by the pri...
media share <= 100%,PASS,0.004,P(media > 100% of sales) <= 0.05,Media cannot explain more than all of sales.
ROAS interval width,WARN,3.070,median relative width <= 3.0,"Wide intervals: show ranges, not a channel ran..."
identifiability,WARN,0.830,share prior-driven <= 0.5,10 of 12 channels are mostly prior-driven; sho...


In [4]:
pd.Series(report_b.channel_status, name="status").sort_values().to_frame()

,status
google_search_brand,prior-data conflict
google_discovery,prior-data conflict
google_display,prior-driven
google_search_nob,prior-driven
facebook_rt,prior-driven
bing_search_brand,prior-driven
bing_shopping_feed,prior-driven
pinterest_viz,prior-driven
pinterest_pr,prior-driven
pinterest_rt,prior-driven


## How this would run in production

1. A scheduled job refits each qualifying account.
2. `evaluate_mmm` grades the fit and stores `report.to_dict()` next to the results.
3. Tier A publishes, Tier B publishes with its warnings shown to the user, Tier C is held back and routed to review.
4. Channel statuses decide how each channel is displayed: an interval for data-driven channels, a "not yet measurable" label for prior-driven ones.
5. Reports are compared across refits to catch regressions, such as a channel moving from data-driven to prior-driven.